In [1]:
!pip install -q transformers datasets accelerate bitsandbytes sentencepiece openpyxl

In [2]:
import sys
print(sys.executable)

/home/pmanandh/llmjudge/bin/python


In [3]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [4]:

import os

os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"

HF_ROOT = "/ediss_data/pmanandh/Thesis/hf_cache"
OUTPUT_DIR = "/ediss_data/pmanandh/Thesis/outputs"

os.environ["HF_HOME"] = HF_ROOT
os.environ["TRANSFORMERS_CACHE"] = f"{HF_ROOT}/transformers"
os.environ["HF_DATASETS_CACHE"] = f"{HF_ROOT}/datasets"
os.environ["HF_HUB_CACHE"] = f"{HF_ROOT}/hub"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{HF_ROOT}/hub"

os.makedirs(HF_ROOT, exist_ok=True)
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

# Create outputs directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import textwrap


import re
import ast
import gc
import json
import shutil
import numpy as np

from tqdm import tqdm
from pathlib import Path

from huggingface_hub import constants



print("=" * 40)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("=" * 40)


print("HF HUB CACHE ACTUAL =", constants.HF_HUB_CACHE)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def delete_model_cache(model_name):
    model_dir = Path(os.environ["TRANSFORMERS_CACHE"]) / f"models--{model_name.replace('/','--')}"
    if model_dir.exists():
        shutil.rmtree(model_dir, ignore_errors=True)

def unique_solution_rate(samples):
    return len(set(filter(None, samples))) / max(len(samples), 1)

def extract_single_function(text, target_function=None, prompt=None, log_path=None):

    if text is None:
        return None

    text = text.replace("\x00", "")
    text = text.replace("Ġ", " ")
    text = text.replace("Ċ", "\n")
    text = text.strip()
    
    # Remove TESTS block completely
    text = re.sub(
        r"\[TESTS\].*?\[/TESTS\]",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # Extract code from fenced Python blocks first
    blocks = re.findall(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if blocks:
        text = blocks[0]
    else:
        text = re.sub(r"```python", "", text)
        text = re.sub(r"```", "", text)


    try:

        markers = [
            "#tests",
            "import unittest",
            "import pytest",
            "class Test",
            "if __name__",
            "<!DOCTYPE",
            "<html",
            "\nprint(",
            "\nassert ",
        ]

        for marker in markers:
            if marker in text:
                text = text.split(marker)[0]
                
        

        print("\nTEXT SENT TO AST:")
        #print(text[:1000])
        

        if target_function:

            match = re.search(
                rf"def\s+{re.escape(target_function)}\s*\(",
                text,
            )

            if not match:
                return None

            text = text[match.start():]

            print(
                f"FOUND TARGET FUNCTION: {target_function}"
            )

        tree = ast.parse(text)

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and (
                    target_function is None
                    or node.name == target_function
                )
            ):

                lines = text.splitlines()

                candidate = "\n".join(
                    lines[node.lineno - 1 : node.end_lineno]
                )

                if is_docstring_only(candidate):
                    
                    print("DOCSTRING ONLY")
                    return None

                return candidate

        return None

    except Exception as e:

        print("\nAST PARSE FAILED")
        print("=" * 80)
        print(repr(e))
        print("=" * 80)
        print(text[:5000])
        print("=" * 80)

        if log_path is not None:

            with open(log_path, "a", encoding="utf-8") as f:

                f.write("\nAST PARSE FAILED\n")
                f.write("=" * 80 + "\n")
                f.write(repr(e))
                f.write("\n\n")
                f.write(text[:5000])
                f.write("\n")

        if target_function:

            match = re.search(
                rf"def\s+{re.escape(target_function)}\s*\(",
                text
            )

            if match:

                recovered = text[match.start():]

                try:
                    
                    recovered = auto_fix_indentation(recovered)

                    tree = ast.parse(recovered)

                    for node in tree.body:

                        if (
                            isinstance(node, ast.FunctionDef)
                            and node.name == target_function
                        ):

                            lines = recovered.splitlines()

                            candidate = "\n".join(
                                lines[
                                    node.lineno - 1 :
                                    node.end_lineno
                                ]
                            )

                            return candidate

                except Exception:
                    pass
        
        return None
        

    
def is_docstring_only(code):

    try:
        
        
        tree = ast.parse(code)

        for node in tree.body:

            if isinstance(node, ast.FunctionDef):

                if len(node.body) == 1:

                    body0 = node.body[0]

                    if (
                        isinstance(body0, ast.Expr)
                        and isinstance(body0.value, ast.Constant)
                        and isinstance(body0.value.value, str)
                    ):
                        return True

        return False

    except:
        return False

def clean_code(code):

    if code is None:
        return None

    code = re.sub(
        r"if __name__ == ['\"]__main__['\"]:.*",
        "",
        code,
        flags=re.DOTALL
    )

    return code.strip()

def indent_body(text):
    return "\n".join(
        "    " + line
        if line.strip()
        else ""
        for line in text.splitlines()
    )

def is_valid_python(code):
    try:
        compile(code, "", "exec")
        return True
    except Exception as e:
        print("\nINVALID PYTHON:")
        print(e)
        print(code)
        return False
    
def contains_target_function(
    code,
    target_function
):
    try:

        tree = ast.parse(code)

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == target_function
            ):
                return True

        return False

    except:
        return False

def clamp(v):
    try: return max(1, min(5, float(v)))
    except: return 1

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_model(model_name):

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map="auto",
        quantization_config=bnb_config
    )

    model.eval()

    return tokenizer, model

def normalize_function_body(body):

    return textwrap.dedent(body).rstrip()

def auto_fix_indentation(code):

    lines = code.splitlines()

    fixed = []
    inside_function = False

    for line in lines:

        stripped = line.strip()

        if stripped.startswith("def "):
            inside_function = True
            fixed.append(line)
            continue

        if inside_function:

            if stripped:
                fixed.append("    " + stripped)
            else:
                fixed.append("")

        else:
            fixed.append(line)

    return "\n".join(fixed)


def judge_code(
    judge_model,
    judge_tokenizer,
    prompt,
    tests,
    code,
):

    eval_prompt = f"""
    You are an expert software engineering evaluator.

    Your task is to assess a candidate solution for a programming problem.

    Problem:
    {prompt}

    Reference Tests:
    {tests}

    Candidate Solution:
    {code}

    Evaluation Rules:

    - Ignore formatting differences.
    - Ignore variable names.
    - Ignore comments.
    - Ignore coding style preferences.
    - Focus on algorithmic quality and likely correctness.

    Scoring Rubric (1-5):

    Correctness:
    1 = Completely incorrect or unrelated solution
    2 = Major logical errors, fails most cases
    3 = Partially correct, solves some cases
    4 = Mostly correct, minor issues or missed edge cases
    5 = Fully correct and likely passes all tests

    Reasoning Quality:
    1 = No meaningful reasoning evident
    2 = Weak or flawed approach
    3 = Adequate approach
    4 = Strong approach with good logic
    5 = Excellent reasoning and problem solving

    Code Clarity:
    1 = Very difficult to understand
    2 = Confusing structure
    3 = Average readability
    4 = Clear and well organized
    5 = Exceptionally clear and maintainable

    Robustness:
    1 = Fragile, fails many edge cases
    2 = Handles few edge cases
    3 = Handles common cases adequately
    4 = Handles most edge cases
    5 = Highly robust and defensive

    Efficiency:
    1 = Extremely inefficient
    2 = Inefficient
    3 = Acceptable efficiency
    4 = Good efficiency
    5 = Optimal or near-optimal efficiency

    Important Instructions:

    - Do NOT assume solutions are either perfect or completely wrong.
    - Use the full scoring range (1, 2, 3, 4, 5).
    - A score of 3 represents an average solution.
    - Reserve a score of 5 only for truly strong solutions.
    - Reserve a score of 1 only for seriously flawed solutions.
    - Consider strengths and weaknesses before scoring.

    First analyze:
    1. Strengths
    2. Weaknesses
    3. Possible edge cases that may fail

    Then assign scores.

    Return ONLY valid JSON:

    {{
        "analysis": {{
            "strengths": "...",
            "weaknesses": "...",
            "edge_cases": "..."
        }},
        "correctness": 1,
        "reasoning": 1,
        "clarity": 1,
        "robustness": 1,
        "efficiency": 1,
        "overall": 1.0,
        "explanation": "brief justification"
    }}
    """

    messages = [
    {
        "role": "user",
        "content": eval_prompt
    }
    ]

    chat_text = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = judge_tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    print("Judge prompt tokens:", inputs["input_ids"].shape[1])

    inputs.pop("token_type_ids", None)

    inputs = {
        k: v.to(judge_model.device)
        for k, v in inputs.items()
    }

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=judge_tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]

    generated = outputs[0][prompt_len:]

    text = judge_tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    print("\n===== GENERATED JUDGE TEXT =====")
    print(repr(text))
    print("================================")

    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1)

    if text.startswith("```"):
        text = text.replace("```", "", 1)

    if text.endswith("```"):
        text = text[:-3]

    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL,
    )

    print("JSON FOUND*", match is not None)

    if not match:

        return {
            "correctness": 0,
            "reasoning": 0,
            "clarity": 0,
            "robustness": 0,
            "efficiency": 0,
            "overall": 0,
            "explanation": text[:1000],
        }

    try:

        result = json.loads(match.group())

        correctness = clamp(result.get("correctness"))
        reasoning = clamp(result.get("reasoning"))
        clarity = clamp(result.get("clarity"))
        robustness = clamp(result.get("robustness"))
        efficiency = clamp(result.get("efficiency"))

        overall = round(
            (
                correctness +
                reasoning +
                clarity +
                robustness +
                efficiency
            ) / 5,
            2,
        )

        return {
            "correctness": correctness,
            "reasoning": reasoning,
            "clarity": clarity,
            "robustness": robustness,
            "efficiency": efficiency,
            "overall": overall,
            "explanation": result.get("explanation", ""),
        }

    except Exception as e:

        print("JSON PARSE FAILED:", e)
        print(text)

        try:

            repaired = match.group()

            repaired = repaired.replace("\n", " ")
            repaired = repaired.replace("\t", " ")

            result = json.loads(repaired)

            correctness = clamp(result.get("correctness"))
            reasoning = clamp(result.get("reasoning"))
            clarity = clamp(result.get("clarity"))
            robustness = clamp(result.get("robustness"))
            efficiency = clamp(result.get("efficiency"))

            overall = round(
                (
                    correctness +
                    reasoning +
                    clarity +
                    robustness +
                    efficiency
                ) / 5,
                2,
            )

            return {
                "correctness": correctness,
                "reasoning": reasoning,
                "clarity": clarity,
                "robustness": robustness,
                "efficiency": efficiency,
                "overall": overall,
                "explanation": result.get(
                    "explanation",
                    ""
                ),
            }

        except Exception:
            pass

        return {
            "correctness": 0,
            "reasoning": 0,
            "clarity": 0,
            "robustness": 0,
            "efficiency": 0,
            "overall": 0,
            "explanation": text[:1000],
        }

        
CHAT_MODELS = [
                    "deepseek-ai/deepseek-coder-6.7b-instruct",
                    "Qwen/Qwen2.5-Coder-7B-Instruct",
                    "codellama/CodeLlama-7b-Instruct-hf",
                    "mistralai/Mistral-7B-Instruct-v0.3",
                ]
num_samples_per_problem = 5
judge_models = ["Qwen/Qwen2.5-Coder-7B-Instruct"]
models_to_test = [
    #"deepseek-ai/deepseek-coder-6.7b-instruct",
    ##"Qwen/Qwen2.5-Coder-7B-Instruct",
    #"bigcode/starcoder2-7b",
    #"codellama/CodeLlama-7b-Instruct-hf",
    "codellama/CodeLlama-7b-Python-hf",
    #"ise-uiuc/Magicoder-S-DS-6.7B",
    ##"aiXcoder/aiXcoder-7B",
    #"mistralai/Mistral-7B-Instruct-v0.3",
    ##"Qwen/Qwen2.5-3B",
]

# Load datasets ONCE
datasets = [
    ("HumanEval", load_dataset("openai/openai_humaneval")["test"]),
    ("HumanEvalNext", load_dataset("AISE-TUDelft/HumanEvalNext")["train"]),
]



PyTorch: 2.5.1+cu121
CUDA: True
CUDA version: 12.1
GPU: Tesla V100-PCIE-16GB
HF HUB CACHE ACTUAL = /ediss_data/pmanandh/Thesis/hf_cache/hub
Device: cuda
GPU: Tesla V100-PCIE-16GB


In [5]:
for test_model_name in models_to_test:

    safe_model_name = (
        test_model_name
        .replace("/", "_")
        .replace("-", "_")
    )

    print("=" * 80)
    print("Loading test model:", test_model_name)

    test_tokenizer, test_model = load_model(
        test_model_name
    )

    for judge_model_name in judge_models:

        print("Judge:", judge_model_name)

        judge_tokenizer, judge_model = load_model(
            judge_model_name
        )

        batch_results = []
        batch_explanations = []

        for dataset_name, dataset in datasets:
            
            metrics_path = (
                f"/ediss_data/pmanandh/Thesis/outputs/"
                f"{dataset_name}_metrics_{safe_model_name}.xlsx"
            )

            explanations_path = (
                f"/ediss_data/pmanandh/Thesis/outputs/"
                f"{dataset_name}_explanations_{safe_model_name}.xlsx"
            )
            
            log_path = (
                    f"/ediss_data/pmanandh/Thesis/outputs/"
                    f"log_{dataset_name}_{safe_model_name}.txt"
                )
            
            with open(log_path, "w", encoding="utf-8") as f:
                f.write(f"MODEL: {test_model_name}\n")
                f.write(f"JUDGE: {judge_models[0]}\n")
                f.write(f"SAMPLES_PER_PROBLEM: {num_samples_per_problem}\n")
                f.write("=" * 100 + "\n")
                
            #dataset = dataset.select(range(2))

            for idx, problem in tqdm(
                enumerate(dataset),
                total=len(dataset),
                desc=dataset_name,
            ):

                prompt = problem["prompt"]
                tests = problem.get("test", "")

                target_function = problem.get(
                    "entry_point",
                    None
                )

                if target_function is None:

                    target_match = re.search(
                        r"def\s+(\w+)\s*\(",
                        prompt
                    )

                    target_function = (
                        target_match.group(1)
                        if target_match
                        else None
                    )



                if test_model_name in CHAT_MODELS:

                    messages = [
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]

                    generate_prompt = test_tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True
                    )

                else:
                    generate_prompt = prompt
                    
                with open(log_path, "a", encoding="utf-8") as f:

                    f.write("\n")
                    f.write("=" * 100 + "\n")
                    f.write(f"DATASET: {dataset_name}\n")
                    f.write(f"PROBLEM: {idx}\n")
                    f.write("=" * 100 + "\n")

                    f.write("\nORIGINAL PROMPT\n")
                    f.write("-" * 50 + "\n")
                    f.write(prompt)
                    f.write("\n")

                    f.write("\nPROMPT SENT TO MODEL\n")
                    f.write("-" * 50 + "\n")
                    f.write(generate_prompt)
                    f.write("\n")

                inputs = test_tokenizer(
                    generate_prompt,
                    return_tensors="pt",
                )

                inputs.pop("token_type_ids", None)

                inputs = {
                    k: v.to(test_model.device)
                    for k, v in inputs.items()
                }

                outputs = test_model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    num_return_sequences=num_samples_per_problem,
                    pad_token_id=test_tokenizer.eos_token_id,
                    eos_token_id=test_tokenizer.eos_token_id,
                )

                candidates = []
                extraction_failed = 0
                function_not_found = 0


                for sample_idx, output in enumerate(outputs):

                    prompt_len = inputs["input_ids"].shape[1]

                    generated_tokens = output[prompt_len:]

                    decoded = test_tokenizer.decode(
                        generated_tokens,
                        skip_special_tokens=True,
                    )

                    decoded = decoded.split("<|im_end|>")[0]
                    decoded = decoded.split("</s>")[0]
                    
                    STOP_MARKERS = [
                        "if __name__",
                        "import doctest",
                        "doctest.testmod",
                        "python_ta.check_all",
                        "unittest",
                        "pytest",
                        "print(",
                    ]

                    for marker in STOP_MARKERS:
                        if marker in decoded:
                            decoded = decoded.split(marker)[0]
                            
                    next_func = re.search(
                        r"\n\s*def\s+\w+\s*\(",
                        decoded
                    )

                    if next_func:
                        decoded = decoded[:next_func.start()]

                    body = normalize_function_body(decoded)
                    
                    BAD_LINES = [
                        "# Do not",
                        "# You can",
                        "# Check",
                    ]

                    cleaned_lines = []

                    for line in body.splitlines():

                        if any(
                            line.strip().startswith(x)
                            for x in BAD_LINES
                        ):
                            continue

                        cleaned_lines.append(line)

                    body = "\n".join(cleaned_lines)

                    # print("BODY REPR:")
                    # print(repr(body))

                    with open(log_path, "a", encoding="utf-8") as f:

                        f.write("\nBODY REPR\n")
                        f.write("-" * 50 + "\n")
                        f.write(repr(body))
                        f.write("\n")

                        f.write("\nNORMALIZED BODY\n")
                        f.write("-" * 50 + "\n")
                        f.write(body)
                        f.write("\n")

                    body = textwrap.dedent(body).strip()

                    normalized_lines = []

                    for line in body.splitlines():

                        if not line.strip():
                            normalized_lines.append("")
                            continue

                        stripped = line.lstrip()

                        indent = len(line) - len(stripped)

                        # normalize indentation to multiples of 4
                        normalized_indent = round(indent / 4) * 4

                        normalized_lines.append(
                            (" " * normalized_indent) + stripped
                        )

                    body = "\n".join(normalized_lines)

                    # print("\nAFTER NORMALIZATION:")
                    # print(repr(body))

                    try:
                        temp_code = (
                            "def temp_func():\n"
                            + textwrap.indent(body, "    ")
                        )
                        ast.parse(temp_code)

                    except Exception:
                        candidates.append(None)
                        extraction_failed += 1
                        continue

                    indented_body = textwrap.indent(
                        body,
                        "    "
                    )

                    candidate = (
                        prompt.rstrip()
                        + "\n"
                        + indented_body
                    )

                    #candidate = auto_fix_indentation(candidate)
                    
                    bad_patterns = [
                        "TODO",
                        "...",
                        "NotImplemented",
                    ]
                    
                    if re.search(r"^\s*pass\s*$", body, re.MULTILINE):
                        candidate = None


                    if any(x in body for x in bad_patterns):
                        candidate = None

                    if candidate is not None and not is_valid_python(candidate):
                        candidate = None

                                        
                    with open(log_path, "a", encoding="utf-8") as f:

                        f.write(f"\nRAW GENERATED OUTPUT | SAMPLE {sample_idx+1}\n")
                        f.write("-" * 50 + "\n")
                        f.write(decoded)
                        f.write("\n")
                    
                    if not decoded.strip():

                        extraction_failed += 1

                        candidates.append(None)

                        continue
                    
                    candidate_text = decoded

                    if candidate is None:
                        extraction_failed += 1
                        
                    
                    
                    with open(log_path, "a", encoding="utf-8") as f:
                        
                    
                        f.write(f"\nCANDIDATE TEXT | SAMPLE {sample_idx+1}\n")
                        f.write("-" * 50 + "\n")
                        f.write(candidate_text)
                        f.write("\n")

                    #print("\nPROMPT:")
                    #print(prompt[:1000])

                    #print("\nCANDIDATE TEXT:")
                    #print(candidate_text[:2000])

                    
                    
                    with open(log_path, "a", encoding="utf-8") as f:

                        f.write(f"\nEXTRACTED FUNCTION | SAMPLE {sample_idx+1}\n")
                        f.write("-" * 50 + "\n")

                        if candidate:
                            f.write(candidate)
                        else:
                            f.write("EXTRACTION FAILED")

                        f.write("\n")


                    #print("FULL CODE:")
                    #print(full_code[:2000])


                    #print("\nEXTRACTED FUNCTION:")
                    #print(candidate)

                    #print("\nRAW GENERATED TEXT:")
                    #print(decoded[:2000])

                    candidates.append(candidate)

                diversity = unique_solution_rate(
                    candidates
                )
                
                successful_extractions = len(
                    [c for c in candidates if c is not None]
                )

                overall_scores = []
                correctness_scores = []
                reasoning_scores = []
                clarity_scores = []
                robustness_scores = []
                efficiency_scores = []
                valid_count = 0
                invalid_python = 0

                for sample_idx, candidate in enumerate(candidates):

                    # print("\n====================")
                    # print("CANDIDATE")
                    # print("====================")
                    # print(candidate)
                    # print("====================")
                    
                    # if candidate is not None:

                    #     if not is_valid_python(candidate):

                    #         repaired = auto_fix_indentation(candidate)

                    #         if is_valid_python(repaired):

                    #             print("AUTO REPAIR SUCCESS")

                    #             candidate = repaired
                                
                    python_valid = (
                        candidate is not None
                        and is_valid_python(candidate)
                    )
                    
                    
                    with open(log_path, "a", encoding="utf-8") as f:
                        f.write(
                            f"\nPYTHON_VALID | SAMPLE {sample_idx+1}: "
                            f"{python_valid}\n"
                        )

                    if candidate is not None and not python_valid:
                        invalid_python += 1

                    valid = python_valid

                    print("VALID:", valid)

                    if not valid:
                        
                        with open(log_path, "a", encoding="utf-8") as f:

                            f.write(f"\nINVALID PYTHON CODE | SAMPLE {sample_idx+1}\n")

                            f.write("-" * 50 + "\n")

                            if candidate:
                                f.write(candidate)

                            f.write("\n")
                            
                        batch_explanations.append({
                          "test_model": test_model_name,
                          "judge_model": judge_model_name,
                          "dataset": dataset_name,
                          "problem_index": idx,
                          "candidate": candidate,
                          "judge_explanation": "Invalid code",
                          "correctness": 0,
                          "reasoning": 0,
                          "clarity": 0,
                          "robustness": 0,
                          "efficiency": 0,
                          "overall": 0,
                        })

                        continue

                    

                    try:
                        
                        gc.collect()

                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                        result = judge_code(
                            judge_model,
                            judge_tokenizer,
                            prompt,
                            tests,
                            candidate,
                        )

                    except torch.cuda.OutOfMemoryError:

                        print("JUDGE OOM")

                        with open(log_path, "a", encoding="utf-8") as f:

                            f.write("\nJUDGE OOM\n")
                            f.write("-" * 50 + "\n")

                            f.write(f"DATASET: {dataset_name}\n")
                            f.write(f"PROBLEM: {idx}\n")
                            f.write(f"SAMPLE: {sample_idx + 1}\n")

                        gc.collect()

                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                            torch.cuda.ipc_collect()

                        result = {
                            "correctness": None,
                            "reasoning": None,
                            "clarity": None,
                            "robustness": None,
                            "efficiency": None,
                            "overall": None,
                            "explanation": "Judge OOM",
                        }

                    if result["overall"] is None:

                        batch_explanations.append({
                            "test_model": test_model_name,
                            "judge_model": judge_model_name,
                            "dataset": dataset_name,
                            "problem_index": idx,
                            "candidate": candidate,
                            "judge_status": "OOM",
                            "judge_explanation": "Judge OOM",
                            "correctness": None,
                            "reasoning": None,
                            "clarity": None,
                            "robustness": None,
                            "efficiency": None,
                            "overall": None,
                        })

                        continue
                    
                    valid_count += 1
                    
                    with open(log_path, "a", encoding="utf-8") as f:

                        f.write(f"\nJUDGE RESULT | SAMPLE {sample_idx+1}\n")
                        f.write("-" * 50 + "\n")

                        f.write(json.dumps(result, indent=2))
                        f.write("\n")

                    #print("RESULT:")
                    #print(result)



                    correctness_scores.append(
                        result["correctness"]
                    )

                    reasoning_scores.append(
                        result["reasoning"]
                    )

                    clarity_scores.append(
                        result["clarity"]
                    )

                    robustness_scores.append(
                        result["robustness"]
                    )

                    efficiency_scores.append(
                        result["efficiency"]
                    )

                    overall_scores.append(
                        result["overall"]
                    )

                    batch_explanations.append({
                        "test_model": test_model_name,
                        "judge_model": judge_model_name,
                        "dataset": dataset_name,
                        "problem_index": idx,
                        "candidate": candidate,
                        "judge_explanation":
                            result["explanation"],
                        **result
                    })

                if valid_count > 0:

                    avg_correctness = np.mean(correctness_scores)
                    avg_reasoning = np.mean(reasoning_scores)
                    avg_clarity = np.mean(clarity_scores)
                    avg_robustness = np.mean(robustness_scores)
                    avg_efficiency = np.mean(efficiency_scores)
                    avg_overall = np.mean(overall_scores)
                    std_overall = np.std(overall_scores)

                    best_overall = max(overall_scores)

                else:

                    avg_correctness = 0
                    avg_reasoning = 0
                    avg_clarity = 0
                    avg_robustness = 0
                    avg_efficiency = 0
                    avg_overall = 0
                    std_overall = 0

                    best_overall = 0

                if valid_count > 0:
                    consistency = max(
                        0,
                        min(
                            1,
                            1 - np.std(overall_scores) / 5
                        )
                    )
                else:
                    consistency = 0


                batch_results.append({

                    "test_model": test_model_name,
                    "judge_model": judge_model_name,
                    "dataset": dataset_name,
                    "problem_index": idx,

                    "valid_count": valid_count,
                    "validity_rate": valid_count / num_samples_per_problem,

                    "avg_correctness": avg_correctness,
                    "avg_reasoning": avg_reasoning,
                    "avg_clarity": avg_clarity,
                    "avg_robustness": avg_robustness,
                    "avg_efficiency": avg_efficiency,
                    "avg_overall": avg_overall,
                    "best_overall": best_overall,
                    "std_overall": std_overall,

                    "diversity": diversity,
                    "judge_consistency": consistency,
                    "extraction_failed": extraction_failed,
                    "function_not_found": function_not_found,
                    "invalid_python": invalid_python,
                    "successful_extractions": successful_extractions,
                    
                    

                })


                # Save after every problem
                pd.DataFrame(batch_results).to_excel(
                    metrics_path,
                    index=False
                )

                pd.DataFrame(batch_explanations).to_excel(
                    explanations_path,
                    index=False
                )

                print(
                    f"Checkpoint saved | "
                    f"{dataset_name} | "
                    f"Problem {idx + 1}"
                )


        #pd.DataFrame(batch_results).to_excel(
        #    metrics_path,
        #    index=False
        #)

        #pd.DataFrame(batch_explanations).to_excel(
        #    explanations_path,
        #    index=False
        #)

        del judge_model
        del judge_tokenizer

        cleanup()

    del test_model
    del test_tokenizer

    cleanup()

print("✅ ALL EVALUATIONS COMPLETED")

Loading test model: codellama/CodeLlama-7b-Python-hf


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Judge: Qwen/Qwen2.5-Coder-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

HumanEval:   0%|          | 0/164 [00:00<?, ?it/s]

...

HumanEvalNext:  99%|█████████▉| 81/82 [2:42:32<02:16, 136.85s/it]

Checkpoint saved | HumanEvalNext | Problem 81
VALID: True
Judge prompt tokens: 2103

===== GENERATED JUDGE TEXT =====
'```json\n{\n    "analysis": {\n        "strengths": "The candidate\'s solution is straightforward and follows the provided grading criteria closely.",\n        "weaknesses": "The solution does not handle rounding to 1 decimal place as specified. Additionally, it does not address the requirement to return an empty string for out-of-bounds inputs.",\n        "edge_cases": "The solution will fail for inputs outside the expected range (i.e., negative numbers, numbers greater than 4.0). It also does not handle the case where the GPA is exactly 4.0."\n    },\n    "correctness": 2,\n    "reasoning": 3,\n    "clarity": 4,\n    "robustness": 2,\n    "efficiency": 3,\n    "overall": 2.5,\n    "explanation": "The solution attempts to map GPA values to letter grades but lacks the necessary logic for rounding and handling out-of-bounds inputs. It also does not cover all edge cases 

HumanEvalNext: 100%|██████████| 82/82 [2:45:17<00:00, 120.95s/it]

Checkpoint saved | HumanEvalNext | Problem 82


✅ ALL EVALUATIONS COMPLETED


In [6]:
print("HF_HOME:", os.environ["HF_HOME"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])

print(os.path.exists(os.environ["TRANSFORMERS_CACHE"]))

HF_HOME: /ediss_data/pmanandh/Thesis/hf_cache
TRANSFORMERS_CACHE: /ediss_data/pmanandh/Thesis/hf_cache/transformers
HF_DATASETS_CACHE: /ediss_data/pmanandh/Thesis/hf_cache/datasets
True


In [7]:
from huggingface_hub import constants
import os

print("HF_HOME =", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE env =", os.environ.get("HF_HUB_CACHE"))
print("HUGGINGFACE_HUB_CACHE env =", os.environ.get("HUGGINGFACE_HUB_CACHE"))

print("Actual HF_HUB_CACHE =", constants.HF_HUB_CACHE)

HF_HOME = /ediss_data/pmanandh/Thesis/hf_cache
HF_HUB_CACHE env = /ediss_data/pmanandh/Thesis/hf_cache/hub
HUGGINGFACE_HUB_CACHE env = /ediss_data/pmanandh/Thesis/hf_cache/hub
Actual HF_HUB_CACHE = /ediss_data/pmanandh/Thesis/hf_cache/hub
